# Capítulo 11 · Probabilidade: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática; não possuem correlação com o mundo real.

A **Parte A** percorre o caso do capítulo, o registro de partos de uma maternidade pública, pelas seis perguntas do comitê. A **Parte B** traz o código e o resultado de cada um dos dez exercícios do capítulo (os enunciados estão no livro). Execute as células em ordem. As saídas do código usam ponto decimal.

## Parte A. O caso

**Dados.** `data/deliveries.csv`: 5.000 linhas, uma por parto, e quatro colunas: `delivery_id`, `cesarean`, `hemorrhage`, `infection` (sim/não, escritos `yes`/`no`). Cada célula leva o número da seção do livro (11.1, 11.2, ...), para você acompanhar o capítulo e o caderno lado a lado.

### 11.1 Introdução: os dados

In [1]:
import pandas as pd

df = pd.read_csv("data/deliveries.csv")

# O tamanho do registro: linhas (partos) e colunas
rows, columns = df.shape
print("O registro tem", rows, "linhas (uma por parto) e", columns, "colunas.")
df.head()                          # as cinco primeiras linhas

O registro tem 5000 linhas (uma por parto) e 4 colunas.


,delivery_id,cesarean,hemorrhage,infection
0,D0001,no,no,no
1,D0002,no,no,no
2,D0003,yes,no,no
3,D0004,no,no,no
4,D0005,yes,no,no


### 11.2 Frações no registro

*Que fração dos partos foi cesárea e que fração teve hemorragia?*

In [2]:
# 1. Pegar todas as linhas: cada linha é um parto
n = len(df)
print("O registro tem", n, "partos.")

# 2. Em cada linha, olhar a coluna: está "yes"?
C = df["cesarean"] == "yes"        # True para cada parto cesárea
H = df["hemorrhage"] == "yes"
I = df["infection"] == "yes"
n_cesarean = C.sum()
print("Dos", n, "partos,", n_cesarean, "foram cesáreas e", n - n_cesarean, "não.")

# 3. Contar as linhas "yes" e dividir por todas as linhas
n_hemorrhage = H.sum()
print("Cesárea:", n_cesarean, "de", n, "partos =", round(100 * n_cesarean / n, 2), "%")
print("Hemorragia:", n_hemorrhage, "de", n, "partos =", round(100 * n_hemorrhage / n, 2), "%")

O registro tem 5000 partos.
Dos 5000 partos, 1313 foram cesáreas e 3687 não.
Cesárea: 1313 de 5000 partos = 26.26 %
Hemorragia: 290 de 5000 partos = 5.8 %


### 11.2.1 Experimento, espaço amostral e evento

In [3]:
# O espaço amostral são os 5.000 registros; os eventos H e C são os registros com hemorragia e com cesárea
# Com todos os registros igualmente prováveis: P(evento) = registros no evento / todos os registros
p_hemorrhage = n_hemorrhage / n
p_cesarean = n_cesarean / n
print("P(H) =", round(p_hemorrhage, 4), "e P(C) =", round(p_cesarean, 4))
print("Chance de a próxima mãe admitida ter hemorragia:", round(100 * p_hemorrhage, 2), "%")

P(H) = 0.058 e P(C) = 0.2626
Chance de a próxima mãe admitida ter hemorragia: 5.8 %


### 11.2.2 Por que muitos casos são necessários em uma análise de dados?

In [4]:
# Ler o registro em ordem: a fração de cesáreas se estabiliza?
for k in [10, 50, 100, 500, 1000, 5000]:
    first_k = C.iloc[:k]           # as k primeiras linhas
    print("Após", k, "linhas:", first_k.sum(), "cesáreas =", round(100 * first_k.mean(), 2), "%")

Após 10 linhas: 5 cesáreas = 50.0 %
Após 50 linhas: 19 cesáreas = 38.0 %
Após 100 linhas: 27 cesáreas = 27.0 %
Após 500 linhas: 123 cesáreas = 24.6 %
Após 1000 linhas: 246 cesáreas = 24.6 %
Após 5000 linhas: 1313 cesáreas = 26.26 %


### 11.3 Hemorragia ou infecção: contar cada mãe uma vez

*Qual a chance de a próxima mãe admitida ter hemorragia ou infecção?*

In [5]:
# 1. Cruzar as duas colunas: cada parto cai em exatamente uma de quatro células
table = pd.crosstab(df["hemorrhage"], df["infection"], margins=True, margins_name="Total")
table.index = ["Sem hemorragia", "Hemorragia", "Total"]      # linhas
table.columns = ["Sem infecção", "Infecção", "Total"]        # colunas
display(table)                                             # mostra a tabela formatada

# 2. Contar as mães com cada complicação e com ambas
n_infection = I.sum()
n_both = (H & I).sum()             # "&" significa "e": as duas ao mesmo tempo
print()
print(n_hemorrhage, "mães tiveram hemorragia e", n_infection, "tiveram infecção.")
print("Somando as duas dá", n_hemorrhage + n_infection, "mas as", n_both, "mães com ambas foram contadas duas vezes.")

,Sem infecção,Infecção,Total
Sem hemorragia,4599,111,4710
Hemorragia,271,19,290
Total,4870,130,5000



290 mães tiveram hemorragia e 130 tiveram infecção.
Somando as duas dá 420 mas as 19 mães com ambas foram contadas duas vezes.


### 11.3.1 Regras para combinar as contagens

In [6]:
# 1. Regra da adição: contar as mães com ambas uma única vez
print("Hemorragia ou infecção:", n_hemorrhage, "+", n_infection, "-", n_both, "=", n_hemorrhage + n_infection - n_both, "mães.")

# 2. Conferir contando diretamente
either = H | I                     # "|" significa "ou": ao menos uma das duas
n_either = either.sum()
print("Contando direto:", n_either, "de", n, "mães =", round(100 * n_either / n, 2), "%")

# 3. Complemento: as mães sem nenhuma das duas
print("Sem nenhuma:", n - n_either, "de", n, "mães =", round(100 * (n - n_either) / n, 2), "%")

Hemorragia ou infecção: 290 + 130 - 19 = 401 mães.
Contando direto: 401 de 5000 mães = 8.02 %
Sem nenhuma: 4599 de 5000 mães = 91.98 %


### 11.4 Infecção dentro de cada grupo de parto

*A infecção é mais frequente após cesárea?*

In [7]:
# 1. Separar os partos por tipo: cesárea e vaginal (não cesárea)
n_vaginal = (~C).sum()             # "~" significa "não"

# 2. Dentro de cada grupo, contar as infecções e dividir pelo tamanho desse grupo
infected_cesarean = (C & I).sum()
infected_vaginal = (~C & I).sum()
print("Infecção entre os partos cesárea:", infected_cesarean, "de", n_cesarean, "=", round(infected_cesarean / n_cesarean, 4))
print("Infecção entre os partos vaginais:", infected_vaginal, "de", n_vaginal, "=", round(infected_vaginal / n_vaginal, 4))

Infecção entre os partos cesárea: 87 de 1313 = 0.0663
Infecção entre os partos vaginais: 43 de 3687 = 0.0117


### 11.4.1 A mesma contagem pode responder a perguntas diferentes

In [8]:
# 1. Uma mãe admitida aqui tem um parto vaginal e, depois disso, uma infecção: dois passos
p_vaginal = n_vaginal / n                                  # passo 1: parto vaginal
p_infection_after_vaginal = infected_vaginal / n_vaginal   # passo 2: infecção, entre os partos vaginais
print("Parto vaginal:", round(100 * p_vaginal, 2), "% das mães; infecção entre elas:", round(100 * p_infection_after_vaginal, 2), "%")
print("Parto vaginal e depois infecção:", round(p_vaginal, 4), "x", round(p_infection_after_vaginal, 4), "=", round(p_vaginal * p_infection_after_vaginal, 4), "(", infected_vaginal, "de", n, ")")

# 2. Outra mãe tem uma cesárea e, depois disso, uma infecção: os dois passos mudam
p_cesarean = n_cesarean / n                                # passo 1: cesárea
p_infection_after_cesarean = infected_cesarean / n_cesarean  # passo 2: infecção, entre as cesáreas
print()
print("Cesárea:", round(100 * p_cesarean, 2), "% das mães; infecção entre elas:", round(100 * p_infection_after_cesarean, 2), "%")
print("Cesárea e depois infecção:", round(p_cesarean, 4), "x", round(p_infection_after_cesarean, 4), "=", round(p_cesarean * p_infection_after_cesarean, 4), "(", infected_cesarean, "de", n, ")")

# 3. Probabilidade total: somar os dois caminhos dá a chance de infecção entre todos os partos
total_probability = p_vaginal * p_infection_after_vaginal + p_cesarean * p_infection_after_cesarean
print()
print("Probabilidade total de infecção:", round(total_probability, 4), "; contagem direta:", n_infection, "de", n, "=", round(n_infection / n, 4))

Parto vaginal: 73.74 % das mães; infecção entre elas: 1.17 %
Parto vaginal e depois infecção: 0.7374 x 0.0117 = 0.0086 ( 43 de 5000 )

Cesárea: 26.26 % das mães; infecção entre elas: 6.63 %
Cesárea e depois infecção: 0.2626 x 0.0663 = 0.0174 ( 87 de 5000 )

Probabilidade total de infecção: 0.026 ; contagem direta: 130 de 5000 = 0.026


### 11.5 Hemorragia e infecção ocorrem de forma independente?

*Saber que a próxima mãe admitida teve hemorragia muda a chance de ela ter infecção?*

In [9]:
# 1. Um identificador é só um rótulo: "identificador na primeira metade do registro" é independente de hemorragia?
number = df["delivery_id"].str[1:].astype(int)       # D0001 -> 1
F = number <= 2500                                    # True para D0001 a D2500
n_first = F.sum()
n_first_hemorrhage = (F & H).sum()
print("Esperado se independentes:", round(n_first / n, 4), "x", round(n_hemorrhage / n, 4), "=", round(n_first / n * n_hemorrhage / n, 4))
print("Observado, os dois juntos:", n_first_hemorrhage, "de", n, "=", round(n_first_hemorrhage / n, 4))
print("Hemorragia na primeira metade:", round(n_first_hemorrhage / n_first, 4), "; na segunda metade:", round((~F & H).sum() / (~F).sum(), 4))

# 2. O contraste: hemorragia e infecção
p_h = n_hemorrhage / n
p_i = n_infection / n
print()
print("Esperado se independentes:", round(p_h, 4), "x", round(p_i, 4), "=", round(p_h * p_i, 6))
print("Observado, as duas juntas:", n_both, "de", n, "=", round(n_both / n, 4))

# 3. Infecção entre as mães com e sem hemorragia
print("Infecção entre as mães com hemorragia:", n_both, "de", n_hemorrhage, "=", round(n_both / n_hemorrhage, 4))
print("Infecção entre as mães sem hemorragia:", (~H & I).sum(), "de", (~H).sum(), "=", round((~H & I).sum() / (~H).sum(), 4))

Esperado se independentes: 0.5 x 0.058 = 0.029
Observado, os dois juntos: 145 de 5000 = 0.029
Hemorragia na primeira metade: 0.058 ; na segunda metade: 0.058

Esperado se independentes: 0.058 x 0.026 = 0.001508
Observado, as duas juntas: 19 de 5000 = 0.0038
Infecção entre as mães com hemorragia: 19 de 290 = 0.0655
Infecção entre as mães sem hemorragia: 111 de 4710 = 0.0236


### 11.6 Entre as infecções, quantas ocorreram após cesáreas?

*Entre as mães com infecção, que fração teve cesárea?*

In [10]:
# 1. A tabela: linhas são o tipo de parto, colunas são a infecção
table = pd.crosstab(df["cesarean"], df["infection"], margins=True, margins_name="Total")
table.index = ["Vaginal", "Cesárea", "Total"]                # linhas
table.columns = ["Sem infecção", "Infecção", "Total"]        # colunas
display(table)                                             # mostra a tabela formatada

# 2. Contando diretamente entre as mães infectadas
print()
print("Entre as", n_infection, "infecções,", infected_cesarean, "vieram depois de cesárea =", round(100 * infected_cesarean / n_infection, 2), "%")

# 3. O mesmo invertendo a condição (Bayes): P(C|I) = P(I|C) P(C) / P(I)
print("Inversão:", round(infected_cesarean / n_cesarean, 4), "x", round(n_cesarean / n, 4), "/", round(total_probability, 4), "=", round((infected_cesarean / n_cesarean) * (n_cesarean / n) / total_probability, 4))

,Sem infecção,Infecção,Total
Vaginal,3644,43,3687
Cesárea,1226,87,1313
Total,4870,130,5000



Entre as 130 infecções, 87 vieram depois de cesárea = 66.92 %
Inversão: 0.0663 x 0.2626 / 0.026 = 0.6692


### 11.7 Escolher prontuários para revisão

*De quantas maneiras se escolhem 20 prontuários e qual a chance de a amostra incluir uma infecção?*

In [11]:
from math import comb, factorial, perm

total = n                          # prontuários no registro
without_infection = n - n_infection

# De quantas maneiras se escolhem 20 prontuários? (a ordem não importa)
sets = comb(total, 20)
print("Há", sets, "conjuntos diferentes de 20 prontuários,", len(str(sets)), "algarismos.")

Há 37735558620382470911680923426535607627307325009655406000 conjuntos diferentes de 20 prontuários, 56 algarismos.


### 11.7.1 A ordem importa?

In [12]:
# Ordens dos 20 prontuários escolhidos, e dos dois primeiros lugares
print("Os 20 prontuários escolhidos podem ser postos em", factorial(20), "ordens; os dois primeiros lugares, em", perm(20, 2))

Os 20 prontuários escolhidos podem ser postos em 2432902008176640000 ordens; os dois primeiros lugares, em 380


### 11.7.2 Usar a amostra sem infecções como complementar

In [13]:
# 1. Chance de ao menos uma infecção = 1 - chance de nenhuma (os 20 entre os prontuários sem infecção)
chance_none = comb(without_infection, 20) / comb(total, 20)
print("Chance de 20 prontuários incluírem ao menos uma infecção:", round(1 - chance_none, 4))

# 2. Menor amostra com ao menos 90%: testar tamanhos 1, 2, 3, ... até a chance de nenhuma ser no máximo 10%
size = 1
while 10 * comb(without_infection, size) > comb(total, size):     # compara números inteiros, sem arredondar
    size = size + 1
for m in [size - 1, size]:
    print("Com", m, "prontuários a chance é", round(1 - comb(without_infection, m) / comb(total, m), 6))
print("Logo", size, "é a menor amostra com ao menos 90%.")

Chance de 20 prontuários incluírem ao menos uma infecção: 0.4102
Com 86 prontuários a chance é 0.898259
Com 87 prontuários a chance é 0.90095
Logo 87 é a menor amostra com ao menos 90%.


## Parte B. Soluções dos exercícios

### 11.9 Exercício 1. Experimento, espaço amostral e probabilidades marginais

In [14]:
# 1. O espaço amostral são os 5.000 registros; os eventos H, C e I são os registros com cada "yes"
print("Registros:", n)
print("P(H) =", n_hemorrhage, "/", n, "=", round(n_hemorrhage / n, 4))
print("P(C) =", n_cesarean, "/", n, "=", round(n_cesarean / n, 4))
print("P(I) =", n_infection, "/", n, "=", round(n_infection / n, 4))

Registros: 5000
P(H) = 290 / 5000 = 0.058
P(C) = 1313 / 5000 = 0.2626
P(I) = 130 / 5000 = 0.026


### 11.9 Exercício 2. Por que muitos casos são necessários

In [15]:
# 1. Fração de hemorragia entre os k primeiros registros
for k in [10, 50, 100, 500, 1000, 5000]:
    first_k = H.iloc[:k]
    print("Após", k, "linhas:", first_k.sum(), "hemorragias =", round(100 * first_k.mean(), 2), "%")

# 2. Mudança de 10 a 100 registros e de 1.000 a 5.000 registros, em pontos percentuais
print()
print("De 10 a 100 registros:", round(100 * (H.iloc[:100].mean() - H.iloc[:10].mean()), 2), "pontos")
print("De 1000 a 5000 registros:", round(100 * (H.mean() - H.iloc[:1000].mean()), 2), "pontos")

Após 10 linhas: 0 hemorragias = 0.0 %
Após 50 linhas: 1 hemorragias = 2.0 %
Após 100 linhas: 7 hemorragias = 7.0 %
Após 500 linhas: 25 hemorragias = 5.0 %
Após 1000 linhas: 47 hemorragias = 4.7 %
Após 5000 linhas: 290 hemorragias = 5.8 %

De 10 a 100 registros: 7.0 pontos
De 1000 a 5000 registros: 1.1 pontos


### 11.9 Exercício 3. OU, E, NÃO e eventos mutuamente exclusivos

In [16]:
# 1. Os quatro grupos: só hemorragia, só infecção, ambas, nenhuma
only_h = (H & ~I).sum()
only_i = (~H & I).sum()
both = (H & I).sum()
neither = (~H & ~I).sum()
print("Só hemorragia:", only_h, "mães =", round(only_h / n, 4))
print("Só infecção:", only_i, "mães =", round(only_i / n, 4))
print("Ambas:", both, "mães =", round(both / n, 4))
print("Nenhuma:", neither, "mães =", round(neither / n, 4))

# 2. Regra da adição para "hemorragia ou infecção", e a contagem direta
print()
print("Regra da adição:", n_hemorrhage + n_infection - both, "mães =", round((n_hemorrhage + n_infection - both) / n, 4))
print("Contagem direta:", (H | I).sum(), "mães")

# 3. Eventos mutuamente exclusivos não podem ocorrer juntos: contar os registros com ambos
print()
print("Cesárea e parto vaginal juntos:", (C & ~C).sum(), "registros")
print("Hemorragia e infecção juntas:", both, "registros")

Só hemorragia: 271 mães = 0.0542
Só infecção: 111 mães = 0.0222
Ambas: 19 mães = 0.0038
Nenhuma: 4599 mães = 0.9198

Regra da adição: 401 mães = 0.0802
Contagem direta: 401 mães

Cesárea e parto vaginal juntos: 0 registros
Hemorragia e infecção juntas: 19 registros


### 11.9 Exercício 4. Frações marginais, conjuntas e condicionais

In [17]:
# 1. As cinco frações: o numerador e o denominador de cada uma
print("(a) infecção entre as cesáreas:", (C & I).sum(), "/", n_cesarean, "=", round((C & I).sum() / n_cesarean, 4))
print("(b) cesárea e infecção, entre todos:", (C & I).sum(), "/", n, "=", round((C & I).sum() / n, 4))
print("(c) infecção, entre todos:", n_infection, "/", n, "=", round(n_infection / n, 4))
print("(d) hemorragia entre as cesáreas:", (C & H).sum(), "/", n_cesarean, "=", round((C & H).sum() / n_cesarean, 4))
print("(e) parto vaginal e infecção, entre todos:", (~C & I).sum(), "/", n, "=", round((~C & I).sum() / n, 4))

(a) infecção entre as cesáreas: 87 / 1313 = 0.0663
(b) cesárea e infecção, entre todos: 87 / 5000 = 0.0174
(c) infecção, entre todos: 130 / 5000 = 0.026
(d) hemorragia entre as cesáreas: 95 / 1313 = 0.0724
(e) parto vaginal e infecção, entre todos: 43 / 5000 = 0.0086


### 11.9 Exercício 5. Chances em dois passos e probabilidade total

In [18]:
# 1. Parto vaginal e depois infecção; cesárea e depois infecção: primeiro passo vezes segundo passo
n_vaginal = (~C).sum()
path_vaginal = (n_vaginal / n) * ((~C & I).sum() / n_vaginal)
path_cesarean = (n_cesarean / n) * ((C & I).sum() / n_cesarean)
print("Parto vaginal e depois infecção:", round(path_vaginal, 4), "(contagem direta:", round((~C & I).sum() / n, 4), ")")
print("Cesárea e depois infecção:", round(path_cesarean, 4), "(contagem direta:", round((C & I).sum() / n, 4), ")")

# 2. Os dois caminhos somados são a probabilidade total de infecção, e sua razão
print()
print("Probabilidade total de infecção:", round(path_vaginal + path_cesarean, 4), "; contagem direta:", round(n_infection / n, 4))
print("Caminho da cesárea dividido pelo caminho vaginal:", round(path_cesarean / path_vaginal, 2))

Parto vaginal e depois infecção: 0.0086 (contagem direta: 0.0086 )
Cesárea e depois infecção: 0.0174 (contagem direta: 0.0174 )

Probabilidade total de infecção: 0.026 ; contagem direta: 0.026
Caminho da cesárea dividido pelo caminho vaginal: 2.02


### 11.9 Exercício 6. Independência

In [19]:
# 1. O = o identificador é um número ímpar: comparar P(O e C) com P(O) x P(C)
O = number % 2 == 1
print("O e C:", (O & C).sum(), "de", n, "=", round((O & C).sum() / n, 4), "; P(O) x P(C) =", round(O.mean() * C.mean(), 4))
print("Cesárea entre identificadores ímpares:", round(C[O].mean(), 4), "; entre pares:", round(C[~O].mean(), 4))

# 2. Cesárea e hemorragia
print()
print("C e H:", (C & H).sum(), "de", n, "=", round((C & H).sum() / n, 4), "; P(C) x P(H) =", round(C.mean() * H.mean(), 4))
print("Hemorragia entre as cesáreas:", round(H[C].mean(), 4), "; entre os partos vaginais:", round(H[~C].mean(), 4))

O e C: 667 de 5000 = 0.1334 ; P(O) x P(C) = 0.1313
Cesárea entre identificadores ímpares: 0.2668 ; entre pares: 0.2584

C e H: 95 de 5000 = 0.019 ; P(C) x P(H) = 0.0152
Hemorragia entre as cesáreas: 0.0724 ; entre os partos vaginais: 0.0529


### 11.9 Exercício 7. Inverter uma condição

In [20]:
# 1. Entre as mães com hemorragia, que fração teve cesárea? Contagem direta
print("Contagem direta:", (C & H).sum(), "de", n_hemorrhage, "=", round((C & H).sum() / n_hemorrhage, 4))

# 2. O mesmo invertendo a condição: P(C|H) = P(H|C) P(C) / P(H)
p_h_given_c = (C & H).sum() / n_cesarean
print("Inversão:", round(p_h_given_c, 4), "x", round(n_cesarean / n, 4), "/", round(n_hemorrhage / n, 4), "=", round(p_h_given_c * (n_cesarean / n) / (n_hemorrhage / n), 4))

# 3. Comparar com P(C), e ver a parcela de partos vaginais entre as infecções
print()
print("P(C) =", round(n_cesarean / n, 4))
print("Partos vaginais entre as infecções:", (~C & I).sum(), "de", n_infection, "=", round((~C & I).sum() / n_infection, 4))

Contagem direta: 95 de 290 = 0.3276
Inversão: 0.0724 x 0.2626 / 0.058 = 0.3276

P(C) = 0.2626
Partos vaginais entre as infecções: 43 de 130 = 0.3308


### 11.9 Exercício 8. Contar uma revisão de 10 prontuários

In [21]:
# 1. Conjuntos de 10 prontuários, e ordens dos 10 prontuários escolhidos
sets_10 = comb(total, 10)
print("Conjuntos de 10 prontuários:", sets_10, "(", len(str(sets_10)), "algarismos )")
print("Ordens dos 10 prontuários escolhidos:", factorial(10))

# 2. Chance de 10 prontuários incluírem ao menos uma infecção (complementar)
print("Chance de ao menos uma infecção em 10 prontuários:", round(1 - comb(without_infection, 10) / comb(total, 10), 4))

# 3. Menor amostra com ao menos 50% de chance
size = 1
while 2 * comb(without_infection, size) > comb(total, size):
    size = size + 1
print("Menor amostra com ao menos 50%:", size, "prontuários (", round(1 - comb(without_infection, size - 1) / comb(total, size - 1), 4), "com", size - 1, ")")

Conjuntos de 10 prontuários: 2667017604016906260066258312000 ( 31 algarismos )
Ordens dos 10 prontuários escolhidos: 3628800
Chance de ao menos uma infecção em 10 prontuários: 0.2318
Menor amostra com ao menos 50%: 27 prontuários ( 0.4968 com 26 )


### 11.9 Exercício 9. Probabilidades marginais alternativas (alt.)

In [22]:
# 1. Marginais alternativas: P(H) = 0,02 e P(I) = 0,03
p_h = 0.02
p_i = 0.03

# 2. Menor e maior valor possível de P(H ou I), e seu valor se H e I fossem independentes
lower = max(p_h, p_i)
upper = min(1, p_h + p_i)
print("P(H ou I) deve estar entre", lower, "e", upper)
print("Se os eventos fossem independentes, P(H ou I) =", round(p_h + p_i - p_h * p_i, 4))

P(H ou I) deve estar entre 0.03 e 0.05
Se os eventos fossem independentes, P(H ou I) = 0.0494


### 11.9 Exercício 10. Devolver prontuários entre sorteios (alt.)

In [23]:
# 1. Chance de ao menos uma infecção em 20 sorteios, com e sem devolver o prontuário
with_replacement = 1 - (without_infection / total) ** 20
without_replacement = 1 - comb(without_infection, 20) / comb(total, 20)
print("Devolvendo cada prontuário:", round(with_replacement, 6), "| sem devolver:", round(without_replacement, 6))

# 2. A diferença entre os dois procedimentos
print("A diferença é de apenas", round(without_replacement - with_replacement, 6))

Devolvendo cada prontuário: 0.409555 | sem devolver: 0.410156
A diferença é de apenas 0.0006


---
*Dados sintéticos (semente 11, `generate_data.py`). Valores apenas didáticos; nada aqui é recomendação clínica nem estimativa para uma unidade real.*